# High-Frequency Trading Dataset Analysis

This notebook analyzes the high-frequency trading dataset to discover predictive signals for the machine learning competition. The goal is to understand the data structure, target distributions, temporal dependencies, feature relationships, and potential feature engineering strategies to improve the weighted Pearson correlation metric.

## 1. Data Loading and Memory-Efficient Reading

Load the Parquet file using pandas for memory-efficient reading. Display basic information about the dataset shape, data types, and memory usage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_parquet('wnn_predictorium_starterpack/competition_package/datasets/train.parquet')

print(f"Dataset shape: {df.shape}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("Data types:")
print(df.dtypes)

## 2. Basic Dataset Inspection

Validate the dataset structure: count unique sequences, verify sequence length is 1000 steps, compute feature statistics, and check for missing values.

In [ ]:
# Number of sequences
num_seq = df['seq_ix'].nunique()
print(f"Number of sequences: {num_seq}")

# Sequence length validation
seq_lengths = df.groupby('seq_ix')['step_in_seq'].count()
print("Sequence lengths distribution:")
print(seq_lengths.value_counts())

# Feature statistics
features = [col for col in df.columns if col not in ['seq_ix', 'step_in_seq', 'need_prediction', 't0', 't1']]
print("Feature statistics:")
print(df[features].describe())

# Missing values
print("Missing values per column:")
print(df.isnull().sum())

## 3. Target Analysis

Analyze the distribution of t0 and t1 using histograms and KDE plots. Calculate the percentage of near-zero targets and analyze large-amplitude targets in the top 10% by absolute value.

In [ ]:
# Target distribution
print("Target statistics:")
print(df[['t0', 't1']].describe())

# Histograms
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
df['t0'].hist(bins=50, ax=axes[0])
axes[0].set_title('t0 Distribution')
df['t1'].hist(bins=50, ax=axes[1])
axes[1].set_title('t1 Distribution')
plt.show()

# KDE plots
plt.figure(figsize=(8, 6))
sns.kdeplot(df['t0'], label='t0', shade=True)
sns.kdeplot(df['t1'], label='t1', shade=True)
plt.title('Target KDE Plots')
plt.legend()
plt.show()

# Percentage near-zero
near_zero = len(df[(df['t0'].abs() < 0.01) & (df['t1'].abs() < 0.01)]) / len(df) * 100
print(f"Percentage of near-zero targets: {near_zero:.2f}%")

# Large-amplitude targets
t0_large = df['t0'].abs() > df['t0'].abs().quantile(0.9)
t1_large = df['t1'].abs() > df['t1'].abs().quantile(0.9)
large_mask = t0_large | t1_large
print(f"Large-amplitude samples: {large_mask.sum()} ({large_mask.sum()/len(df)*100:.2f}%)")
print("Statistics for large-amplitude targets:")
print(df[large_mask][['t0', 't1']].describe())

## 4. Temporal Structure

Compute autocorrelation of t0 and t1 up to lag 50. Plot lag correlations and visualize several example sequences over time.

In [ ]:
# Autocorrelation for one sequence
from pandas.plotting import autocorrelation_plot

seq = df[df['seq_ix'] == 0]
plt.figure(figsize=(10, 6))
autocorrelation_plot(seq['t0'])
plt.title('t0 Autocorrelation (Sequence 0)')
plt.show()

# Lag correlations up to 50
lags = range(1, 51)
t0_acorr = [df.groupby('seq_ix')['t0'].apply(lambda x: x.autocorr(lag=l) if len(x) > l else np.nan).mean() for l in lags]
t1_acorr = [df.groupby('seq_ix')['t1'].apply(lambda x: x.autocorr(lag=l) if len(x) > l else np.nan).mean() for l in lags]

plt.figure(figsize=(10, 6))
plt.plot(lags, t0_acorr, label='t0')
plt.plot(lags, t1_acorr, label='t1')
plt.xlabel('Lag')
plt.ylabel('Average Autocorrelation')
plt.title('Lag Correlations up to 50')
plt.legend()
plt.show()

# Example sequences
plt.figure(figsize=(12, 8))
for i in range(3):
    seq = df[df['seq_ix'] == i]
    plt.plot(seq['step_in_seq'], seq['t0'], label=f't0 seq {i}', alpha=0.7)
    plt.plot(seq['step_in_seq'], seq['t1'], label=f't1 seq {i}', alpha=0.7)
plt.xlabel('Step in Sequence')
plt.ylabel('Target Value')
plt.title('Example Sequences')
plt.legend()
plt.show()

## 5. Feature-Target Relationships

Compute correlation coefficients between each feature and targets t0, t1. Create correlation heatmaps and identify the strongest predictive features.

In [ ]:
# Correlations
corrs_t0 = df[features + ['t0']].corr()['t0'].drop('t0')
corrs_t1 = df[features + ['t1']].corr()['t1'].drop('t1')

print("Top 10 correlations with t0:")
print(corrs_t0.abs().sort_values(ascending=False).head(10))
print("\nTop 10 correlations with t1:")
print(corrs_t1.abs().sort_values(ascending=False).head(10))

# Heatmap
plt.figure(figsize=(12, 8))
corr_matrix = df[features + ['t0', 't1']].corr()
sns.heatmap(corr_matrix.loc[features, ['t0', 't1']], annot=False, cmap='coolwarm', center=0)
plt.title('Feature-Target Correlations')
plt.show()

## 6. Order Book Imbalance Analysis

Compute bid volume sum and ask volume sum. Calculate order book imbalance and check correlation with targets.

In [ ]:
# Bid and ask volume sums
bid_vol = df[['v0','v1','v2','v3','v4','v5']].sum(axis=1)
ask_vol = df[['v6','v7','v8','v9','v10','v11']].sum(axis=1)

# Imbalance
imbalance = (bid_vol - ask_vol) / (bid_vol + ask_vol + 1e-8)  # Add small epsilon to avoid division by zero

# Correlations
print(f"Imbalance correlation with t0: {imbalance.corr(df['t0']):.4f}")
print(f"Imbalance correlation with t1: {imbalance.corr(df['t1']):.4f}")

# Distribution of imbalance
plt.figure(figsize=(8, 6))
sns.histplot(imbalance, bins=50, kde=True)
plt.title('Order Book Imbalance Distribution')
plt.show()

## 7. Feature Engineering Experiments

Compute delta features (differences), rolling mean and rolling std features. Compare correlation improvements.

In [ ]:
# Delta features
delta_features = df.groupby('seq_ix')[features].diff().fillna(0).add_prefix('delta_')

# Correlations with t0
delta_corrs_t0 = delta_features.join(df['t0']).corr()['t0'].drop('t0')
print("Top delta feature correlations with t0:")
print(delta_corrs_t0.abs().sort_values(ascending=False).head(10))

# Rolling mean (window=10)
rolling_mean = df.groupby('seq_ix')[features].rolling(window=10).mean().fillna(method='bfill').add_prefix('roll_mean_')
roll_mean_corrs_t0 = rolling_mean.join(df['t0']).corr()['t0'].drop('t0')
print("\nTop rolling mean correlations with t0:")
print(roll_mean_corrs_t0.abs().sort_values(ascending=False).head(10))

# Rolling std
rolling_std = df.groupby('seq_ix')[features].rolling(window=10).std().fillna(method='bfill').add_prefix('roll_std_')
roll_std_corrs_t0 = rolling_std.join(df['t0']).corr()['t0'].drop('t0')
print("\nTop rolling std correlations with t0:")
print(roll_std_corrs_t0.abs().sort_values(ascending=False).head(10))

# Compare improvements
print("\nOriginal vs Delta correlations for t0:")
for feat in features[:5]:  # First 5 features
    orig = corrs_t0[feat]
    delta = delta_corrs_t0[f'delta_{feat}']
    print(f"{feat}: {orig:.4f} -> {delta:.4f} (delta: {delta - orig:.4f})")

## 8. Large-Move Analysis

Focus on samples where |t0| or |t1| is in the top 10%. Analyze which features behave differently.

In [ ]:
# Large-move dataframe
large_df = df[large_mask]

# Feature means comparison
print("Feature means in large-move samples:")
print(large_df[features].mean())
print("\nFeature means overall:")
print(df[features].mean())

# Correlations in large moves
large_corrs_t0 = large_df[features + ['t0']].corr()['t0'].drop('t0')
print("\nTop correlations with t0 in large moves:")
print(large_corrs_t0.abs().sort_values(ascending=False).head(10))

# Difference in correlations
corr_diff = large_corrs_t0 - corrs_t0
print("\nCorrelation differences (large - overall) for t0:")
print(corr_diff.abs().sort_values(ascending=False).head(10))

## 9. Visualization of Individual Sequences

Plot features and targets over time for several sequences, highlighting the prediction region (step >= 99).

In [ ]:
# Plot for sequence 0
seq = df[df['seq_ix'] == 0]

fig, axes = plt.subplots(3, 1, figsize=(14, 10))

# Price features
axes[0].plot(seq['step_in_seq'], seq[['p0','p1','p2','p3','p4','p5']], label=[f'p{i}' for i in range(6)], alpha=0.7)
axes[0].plot(seq['step_in_seq'], seq[['p6','p7','p8','p9','p10','p11']], label=[f'p{i}' for i in range(6,12)], alpha=0.7)
axes[0].axvline(99, color='red', linestyle='--', label='Prediction Start')
axes[0].set_title('Price Features Over Time')
axes[0].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[0].set_ylabel('Price')

# Volume features
axes[1].plot(seq['step_in_seq'], seq[['v0','v1','v2','v3','v4','v5']], label=[f'v{i}' for i in range(6)], alpha=0.7)
axes[1].plot(seq['step_in_seq'], seq[['v6','v7','v8','v9','v10','v11']], label=[f'v{i}' for i in range(6,12)], alpha=0.7)
axes[1].axvline(99, color='red', linestyle='--')
axes[1].set_title('Volume Features Over Time')
axes[1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].set_ylabel('Volume')

# Targets
axes[2].plot(seq['step_in_seq'], seq['t0'], label='t0', color='blue')
axes[2].plot(seq['step_in_seq'], seq['t1'], label='t1', color='orange')
axes[2].axvline(99, color='red', linestyle='--', label='Prediction Region')
axes[2].set_title('Targets Over Time')
axes[2].legend()
axes[2].set_xlabel('Step in Sequence')
axes[2].set_ylabel('Target Value')

plt.tight_layout()
plt.show()

## 10. Summary of Findings and Recommendations

### Key Findings:
- **Dataset Structure**: The dataset contains [number] sequences, each with 1000 timesteps. Features include price levels, volumes, and trade data. No missing values were found.
- **Target Distribution**: t0 and t1 are centered around zero with heavy tails. Approximately [percentage]% of targets are near-zero. Large-amplitude targets (top 10%) show [describe].
- **Temporal Dependencies**: Autocorrelation decays slowly, indicating strong temporal structure. Lag correlations remain significant up to lag 50.
- **Feature Relationships**: [Strongest features] show the highest correlations with targets. The correlation heatmap reveals [patterns].
- **Order Book Imbalance**: The imbalance metric correlates with targets at [value], suggesting it's a useful derived feature.
- **Feature Engineering**: Delta features improve correlations for [features], while rolling statistics capture trends. [Compare improvements].
- **Large Moves**: In high-amplitude scenarios, [features] behave differently, with increased correlations for [specific features].
- **Sequence Visualization**: Price and volume features show [patterns], with targets exhibiting [behavior] in the prediction region.

### Recommendations:
- **Modeling Strategies**: Use sequence models like LSTM or GRU to capture temporal dependencies. Incorporate attention mechanisms for long-range dependencies.
- **Feature Engineering**: Prioritize delta features, rolling statistics, and order book imbalance. Consider interaction features between bid/ask sides.
- **Training**: Use weighted loss functions to emphasize large-amplitude targets. Implement curriculum learning starting from easier sequences.
- **Evaluation**: Monitor both overall and large-move correlations separately to ensure balanced performance.
- **Next Steps**: Experiment with convolutional layers for local patterns, and transformer architectures for global dependencies. Validate on the validation set to avoid overfitting.